# Voiceoftaade — MoneyPrinterTurbo Colab Launcher

This notebook launches the MoneyPrinterTurbo Streamlit dashboard in an existing Google Colab runtime and exposes it through an email-protected Cloudflare Quick Tunnel.

## Before you run it
- Run the installation/setup cell from the official MoneyPrinterTurbo Colab notebook first.
- This launcher expects the project at `/content/MoneyPrinterTurbo` and Streamlit on port `8501`.
- When prompted, enter the email address that should be allowed to access the dashboard. Cloudflare sends that address a one-time PIN.
- Do not share the tunnel URL. Stop the tunnel when you finish.
- Colab runtimes are temporary; the tunnel URL changes when restarted.

The tunnel is for temporary testing, not production hosting. This notebook does not disable Streamlit's CORS or XSRF protections and does not use ngrok.

In [ ]:
import re
import subprocess
import time
from pathlib import Path
from urllib.request import urlopen, urlretrieve
from urllib.error import URLError
from getpass import getpass

REPO_DIR = Path("/content/MoneyPrinterTurbo")
PORT = 8501
APP_LOG = Path("/content/moneyprinterturbo-webui.log")
TUNNEL_LOG = Path("/content/cloudflared-quick-tunnel.log")
CLOUDFLARED = Path("/usr/local/bin/cloudflared")

if not (REPO_DIR / "webui" / "Main.py").is_file():
    raise RuntimeError(
        "MoneyPrinterTurbo is not installed at /content/MoneyPrinterTurbo. "
        "Run the installation cell in the official Colab notebook first."
    )

# Install the latest Linux x86-64 cloudflared binary if needed.
if not CLOUDFLARED.exists():
    download_path = Path("/content/cloudflared-linux-amd64")
    print("Downloading cloudflared...")
    urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        download_path,
    )
    download_path.replace(CLOUDFLARED)
    CLOUDFLARED.chmod(0o755)

version = subprocess.run(
    [str(CLOUDFLARED), "--version"],
    check=True,
    capture_output=True,
    text=True,
)
print(version.stdout.strip())

help_result = subprocess.run(
    [str(CLOUDFLARED), "tunnel", "--help"],
    capture_output=True,
    text=True,
)
if "--allowed-mail" not in help_result.stdout + help_result.stderr:
    raise RuntimeError(
        "This cloudflared build does not support email protection. "
        "The tunnel was NOT started; update cloudflared before continuing."
    )

# Reuse the existing Streamlit app if it is already healthy.
def app_is_healthy():
    try:
        with urlopen(f"http://127.0.0.1:{PORT}/_stcore/health", timeout=3) as response:
            return response.status == 200
    except (URLError, TimeoutError, OSError):
        return False

if not app_is_healthy():
    app_log_handle = APP_LOG.open("w", encoding="utf-8")
    streamlit_proc = subprocess.Popen(
        [
            "uv", "run", "streamlit", "run", "webui/Main.py",
            f"--server.port={PORT}",
            "--server.address=0.0.0.0",
            "--browser.gatherUsageStats=False",
            "--client.toolbarMode=minimal",
            "--server.showEmailPrompt=False",
        ],
        cwd=REPO_DIR,
        stdout=app_log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )

    deadline = time.time() + 90
    while time.time() < deadline and not app_is_healthy():
        if streamlit_proc.poll() is not None:
            break
        time.sleep(2)

    if not app_is_healthy():
        app_log_handle.flush()
        recent = APP_LOG.read_text(encoding="utf-8", errors="replace")[-4000:]
        raise RuntimeError("Streamlit did not become ready. Recent log:\n" + recent)

print("MoneyPrinterTurbo is responding on port", PORT)

# Stop a tunnel started earlier by this same notebook, if present.
previous_tunnel = globals().get("cloudflared_proc")
if previous_tunnel is not None and previous_tunnel.poll() is None:
    previous_tunnel.terminate()
    previous_tunnel.wait(timeout=10)

allowed_email = getpass(
    "Email address that should be allowed to open the dashboard: "
).strip()
if not re.fullmatch(r"[^\s@]+@[^\s@]+\.[^\s@]+", allowed_email):
    raise ValueError("Please enter a valid email address. No tunnel was started.")

tunnel_log_handle = TUNNEL_LOG.open("w", encoding="utf-8")
cloudflared_proc = subprocess.Popen(
    [
        str(CLOUDFLARED), "tunnel", "--no-autoupdate",
        "--url", f"http://127.0.0.1:{PORT}",
        "--allowed-mail", allowed_email,
    ],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Starting the protected tunnel...")
deadline = time.time() + 60
tunnel_url = None

while time.time() < deadline:
    tunnel_log_handle.flush()
    log_text = TUNNEL_LOG.read_text(encoding="utf-8", errors="replace")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        tunnel_url = match.group(0)
        break
    if cloudflared_proc.poll() is not None:
        break
    time.sleep(2)

if not tunnel_url:
    tunnel_log_handle.flush()
    recent = TUNNEL_LOG.read_text(encoding="utf-8", errors="replace")[-4000:]
    raise RuntimeError(
        "The protected tunnel did not start. Recent log:\n" + recent
    )

print("\nDashboard address:", tunnel_url)
print("Access is restricted to the email address entered above.")
print("Cloudflare will send that address a one-time PIN when the link is opened.")
print("Keep this Colab runtime running. Stop the tunnel when finished.")


## Stop the tunnel

Run the cell below when you are finished. The Streamlit app can remain running until the Colab runtime is closed.

In [ ]:
proc = globals().get('cloudflared_proc')
if proc is not None and proc.poll() is None:
    proc.terminate()
    proc.wait(timeout=10)
    print('Tunnel stopped.')
else:
    print('No active tunnel process found.')
